In [2]:
import pandas as pd, polars as pl, matplotlib.pyplot as plt
import os, subprocess

In [6]:
import datetime as dt

# Data reading

## load all 2024 months of taxi rides

In [3]:
# i am reusing loads used in exercises because they have nice way to prevent from 
#repeated redownload 
DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    os.mkdir(DATA_DIR)


for month in range(1, 13):
    if not os.path.exists(f"{DATA_DIR}/{month}.parquet"):
        subprocess.run(
            [
                "wget",
                f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{month:02}.parquet",
                "-O",
                f"{DATA_DIR}/{month}.parquet",
            ]
        )


total_size = sum(
    os.path.getsize(f"{DATA_DIR}/{month}.parquet") for month in range(1, 13)
)  # bytes
total_size_mb = total_size // (1024 * 1024)
print(f"Total dataset size: {total_size_mb} MB")

Total dataset size: 660 MB


##    - also load taxi zone lookup data


In [4]:
#also i am borroing this one
if not os.path.exists(f"{DATA_DIR}/taxi_zone_lookup.csv"):
    subprocess.run(
        [
            "wget",
            "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
            "-O",
            f"{DATA_DIR}/taxi_zone_lookup.csv",
        ]
    )

## include only rides starting in 2024 and ending at most at 01.01.2025

In [5]:
df = pl.read_parquet(f"{DATA_DIR}/1.parquet")

In [7]:
df = (
    df.filter
    (
        (pl.col("tpep_pickup_datetime").dt.year() == 2024)
        & (pl.col("tpep_dropoff_datetime").dt.date() <= dt.date(2025,1,1))
    )

)

## ptimize data types, particularly for integers and categorical strings

In [8]:
df.describe()

statistic,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
str,f64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",2.964609e6,"""2964609""","""2964609""",2.824447e6,2.964609e6,2.824447e6,"""2824447""",2.964609e6,2.964609e6,2.964609e6,2.964609e6,2.964609e6,2.964609e6,2.964609e6,2.964609e6,2.964609e6,2.964609e6,2.824447e6,2.824447e6
"""null_count""",0.0,"""0""","""0""",140162.0,0.0,140162.0,"""140162""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,140162.0,140162.0
"""mean""",1.754203,"""2024-01-17 01:02:09.401078""","""2024-01-17 01:17:46.178878""",1.339279,3.652171,2.069365,null,166.01786,165.116522,1.161268,18.175072,1.451595,0.483383,3.335878,0.527022,0.975632,26.801522,2.256126,0.141161
"""std""",0.432591,null,null,0.850279,225.463143,9.823245,null,63.623961,69.31536,0.580867,18.949561,1.804098,0.117759,3.896555,2.128311,0.218362,23.385594,0.823267,0.487624
"""min""",1.0,"""2024-01-01 00:00:00""","""2024-01-01 00:02:42""",0.0,0.0,1.0,"""N""",1.0,1.0,0.0,-899.0,-7.5,-0.5,-80.0,-80.0,-1.0,-900.0,-2.5,-1.75
"""25%""",2.0,"""2024-01-09 15:59:24""","""2024-01-09 16:16:28""",1.0,1.0,1.0,null,132.0,114.0,1.0,8.6,0.0,0.5,1.0,0.0,1.0,15.38,2.5,0.0
"""50%""",2.0,"""2024-01-17 10:45:42""","""2024-01-17 11:03:57""",1.0,1.68,1.0,null,162.0,162.0,1.0,12.8,1.0,0.5,2.7,0.0,1.0,20.1,2.5,0.0
"""75%""",2.0,"""2024-01-24 18:23:56""","""2024-01-24 18:40:30""",1.0,3.11,1.0,null,234.0,234.0,1.0,20.5,2.5,0.5,4.12,0.0,1.0,28.56,2.5,0.0
"""max""",6.0,"""2024-02-01 00:01:15""","""2024-02-02 13:56:52""",9.0,312722.3,99.0,"""Y""",265.0,265.0,4.0,5000.0,14.25,4.0,428.0,115.92,1.0,5000.0,2.5,1.75


In [12]:
df = df.with_columns(
    pl.col("VendorID").cast(pl.Int8),
    pl.col("tpep_pickup_datetime").cast(pl.Datetime),
    pl.col("tpep_dropoff_datetime").cast(pl.Datetime),
    pl.col("passenger_count").cast(pl.Int32),
    pl.col("trip_distance").cast(pl.Float32),
    pl.col("RatecodeID").cast(pl.Int8),
    pl.col("store_and_fwd_flag").cast(pl.Utf8),
    pl.col("PULocationID").cast(pl.Int16),
    pl.col("DOLocationID").cast(pl.Int16),
    pl.col("payment_type").cast(pl.Int8),
    pl.col("fare_amount").cast(pl.Float32),
    pl.col("extra").cast(pl.Float32),
    pl.col("mta_tax").cast(pl.Float32),
    pl.col("tip_amount").cast(pl.Float32),
    pl.col("tolls_amount").cast(pl.Float32),
    pl.col("improvement_surcharge").cast(pl.Float32),
    pl.col("total_amount").cast(pl.Float32),
    pl.col("congestion_surcharge").cast(pl.Float32),
    pl.col("Airport_fee").cast(pl.Float32)
)

# Data cleaning and filtering

## all steps in one until it crashes and burns


In [19]:
df = (
    df.lazy().with_columns(
    pl.col("passenger_count").fill_null(1)
    ).filter(
        pl.col("passenger_count") > 0
    ).with_columns(
        (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")
         ).dt.total_minutes().alias("trip_duration")
    ).filter(
        pl.col("trip_duration") <=120
    ).with_columns(
        pl.col("passenger_count").clip(upper_bound=6), #thats actually nice function
        pl.col([
            'fare_amount',
            'extra',
            'mta_tax',
            'tip_amount',
            'tolls_amount',
            'improvement_surcharge',
            'total_amount',
            'congestion_surcharge',
            'Airport_fee']).abs()
    ).filter(
        pl.col('fare_amount') < 1000,
        pl.col('extra') < 1000,
        pl.col('mta_tax') < 1000,
        pl.col('tip_amount') < 1000,
        pl.col('tolls_amount') < 1000,
        pl.col('improvement_surcharge') < 1000,
        pl.col('total_amount') < 1000,
        pl.col('congestion_surcharge') < 1000,
        pl.col('Airport_fee') < 1000
    ).filter(
        pl.col("VendorID").is_not_null(),
        pl.col("VendorID").is_in([1, 2,6,7]), #apparently these are viable
        pl.col('RatecodeID').is_not_null(),
        pl.col('RatecodeID').is_in([1, 2, 3, 4, 5, 6, 99])
    ).collect()
)


In [20]:
df.describe()

statistic,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration
str,f64,str,str,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",2.790279e6,"""2790279""","""2790279""",2.790279e6,2.790279e6,2.790279e6,"""2790279""",2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6,2.790279e6
"""null_count""",0.0,"""0""","""0""",0.0,0.0,0.0,"""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",1.767292,"""2024-01-17 01:19:10.908852""","""2024-01-17 01:33:56.329494""",1.35406,3.253802,2.062026,null,166.214695,165.469493,1.218494,18.5804,1.521325,0.494884,3.429802,0.548861,0.9997,27.484175,2.307441,0.147693,14.268585
"""std""",0.422557,null,null,0.842074,12.18727,9.787474,null,63.185141,69.190398,0.532769,17.849138,1.790384,0.050475,3.911938,2.139818,0.016619,22.282909,0.66657,0.486466,11.79158
"""min""",1.0,"""2024-01-01 00:00:00""","""2024-01-01 00:02:42""",1.0,0.0,1.0,"""N""",1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-13.0
"""25%""",2.0,"""2024-01-09 15:29:06""","""2024-01-09 15:45:50""",1.0,1.0,1.0,null,132.0,114.0,1.0,8.6,0.0,0.5,1.0,0.0,1.0,15.4,2.5,0.0,7.0
"""50%""",2.0,"""2024-01-17 10:35:20""","""2024-01-17 10:53:14""",1.0,1.67,1.0,null,162.0,162.0,1.0,12.8,1.0,0.5,2.8,0.0,1.0,20.120001,2.5,0.0,11.0
"""75%""",2.0,"""2024-01-24 18:43:41""","""2024-01-24 18:58:14""",1.0,3.1,1.0,null,234.0,234.0,1.0,19.799999,2.5,0.5,4.2,0.0,1.0,28.559999,2.5,0.0,18.0
"""max""",2.0,"""2024-02-01 00:01:15""","""2024-02-01 01:10:21""",6.0,15400.320312,99.0,"""Y""",265.0,265.0,4.0,820.0,14.25,4.0,428.0,115.919998,1.0,821.0,2.5,1.75,120.0
